# LeetCode 1644 — Lowest Common Ancestor of a Binary Tree II bằng Python

Link đề: https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree-ii/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree

> **Lưu ý (bài Premium 🔒):** đề trong notebook được viết lại theo hiểu biết về bài (nguồn tham khảo: bản mô tả của doocs/leetcode). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, x):
#         self.val = x
#         self.left = None
#         self.right = None

class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        ...
```

Khác với bài gốc **236**, ở đây `p` hoặc `q` **có thể không có trong cây** — khi đó phải trả về `None`. Notebook đi từ cách **vét cạn kiểm tra mọi nút** → **so sánh hai đường đi từ gốc** → **LCA 236 + kiểm tra tồn tại** → **một lần DFS hậu thứ tự đếm số nút tìm thấy** (trả lời follow-up) → bản **lặp không đệ quy** dùng bảng cha.

## 1. Mục tiêu học

1. Nắm định nghĩa **tổ tiên chung thấp nhất (LCA)**: nút sâu nhất nhận cả `p` và `q` làm hậu duệ (một nút là hậu duệ của chính nó).
2. Hiểu vì sao lời giải chuẩn của bài 236 **sai** khi `p` hoặc `q` vắng mặt, và sửa bằng cách **duyệt hết cây** thay vì dừng sớm.
3. Viết DFS **hậu thứ tự** trả về "số nút mục tiêu tìm thấy trong cây con" và chốt LCA ở nút đầu tiên đạt đủ 2.
4. Viết lời giải **lặp** (BFS + bảng cha) để tránh tràn đệ quy với cây lệch `10⁴` nút.
5. Tự dựng cây từ mảng level-order kiểu LeetCode, sinh cây ngẫu nhiên và đối chiếu nhiều cách.

## 2. Đề bài & ví dụ

Cho gốc `root` của một cây nhị phân và hai nút `p`, `q`. Trả về **tổ tiên chung thấp nhất (LCA)** của `p` và `q`. **Nếu `p` hoặc `q` không tồn tại trong cây, trả về `None`** (`null`). Mọi giá trị nút đều **khác nhau**.

Theo Wikipedia: "LCA của hai nút `p`, `q` trong cây `T` là nút **thấp nhất** có cả `p` và `q` là hậu duệ (cho phép một nút là hậu duệ của chính nó)."

Cây dùng cho cả 3 ví dụ: `root = [3,5,1,6,2,0,8,null,null,7,4]`

```
            3
          /   \
         5     1
        / \   / \
       6   2 0   8
          / \
         7   4
```

**Ví dụ 1:** `p = 5, q = 1` → `3`.

**Ví dụ 2:** `p = 5, q = 4` → `5` (nút có thể là hậu duệ của chính nó).

**Ví dụ 3:** `p = 5, q = 10` → `null` (nút 10 không có trong cây).

**Ràng buộc:**
- Số nút trong khoảng `[1, 10⁴]`.
- `-10⁹ ≤ Node.val ≤ 10⁹`; mọi `Node.val` khác nhau; `p != q`.

**Follow-up:** có thể tìm LCA khi duyệt cây **mà không cần kiểm tra riêng** sự tồn tại của các nút không?

In [1]:
import random
import sys
import time
from collections import deque
from typing import Optional, List

class TreeNode:
    def __init__(self, x):
        self.val = x
        self.left = None
        self.right = None
    def __repr__(self):
        return f"TreeNode({self.val})"

def build_tree(values):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); queue.append(node.right)
        i += 1
    return root

def to_list(root):
    """Cây -> mảng level-order (bỏ các None ở cuối)."""
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val)
        queue.append(node.left); queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    """Tìm nút có giá trị val (lặp, không đệ quy); None nếu không có."""
    stack = [root] if root else []
    while stack:
        node = stack.pop()
        if node.val == val:
            return node
        if node.left: stack.append(node.left)
        if node.right: stack.append(node.right)
    return None

def get_node(root, val):
    """Như LeetCode: nếu val có trong cây thì trả nút thật, ngược lại tạo nút rời."""
    return find(root, val) or TreeNode(val)

TREE = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
examples = [(TREE, 5, 1, 3), (TREE, 5, 4, 5), (TREE, 5, 10, None)]
root = build_tree(TREE)
print("Khứ hồi:", to_list(root))
for vals, pv, qv, exp in examples:
    print(f"p={pv}, q={qv} -> kỳ vọng {exp} | p trong cây? {find(root, pv) is not None}, q trong cây? {find(root, qv) is not None}")

Khứ hồi: [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
p=5, q=1 -> kỳ vọng 3 | p trong cây? True, q trong cây? True
p=5, q=4 -> kỳ vọng 5 | p trong cây? True, q trong cây? True
p=5, q=10 -> kỳ vọng None | p trong cây? True, q trong cây? False


## 3. Kiến thức nền

### 3.1 Cây nhị phân và các kiểu duyệt DFS

- **Tiền thứ tự (preorder):** nút → trái → phải.
- **Trung thứ tự (inorder):** trái → nút → phải.
- **Hậu thứ tự (postorder):** trái → phải → nút. Kiểu này cho phép nút **dùng kết quả của hai cây con** — chính là thứ ta cần cho LCA.

In [2]:
def preorder(node):
    return [] if node is None else [node.val] + preorder(node.left) + preorder(node.right)

def postorder(node):
    return [] if node is None else postorder(node.left) + postorder(node.right) + [node.val]

print("preorder :", preorder(root))
print("postorder:", postorder(root))

preorder : [3, 5, 6, 2, 7, 4, 1, 0, 8]
postorder: [6, 7, 4, 2, 5, 0, 8, 1, 3]


### 3.2 Vẽ cây ra màn hình

Một hàm in cây nằm ngang (gốc bên trái, nhánh phải ở trên) giúp kiểm tra cây dựng đúng chưa.

In [3]:
def show(node, prefix="", is_left=None):
    if node is None:
        return
    show(node.right, prefix + ("│   " if is_left else "    "), False)
    print(prefix + ("└── " if is_left else ("┌── " if is_left is False else "")) + str(node.val))
    show(node.left, prefix + ("    " if is_left else "│   "), True)

show(root)

        ┌── 8
    ┌── 1
    │   └── 0
3
│   │       ┌── 4
│   │   ┌── 2
│   │   │   └── 7
│   └── 5
│       └── 6


### 3.3 So sánh nút bằng `is` hay bằng `val`?

`a is b` kiểm tra **cùng một đối tượng**. Vì đề bảo đảm giá trị **khác nhau**, so sánh `a.val == b.val` cũng xác định duy nhất một nút. Trong notebook ta so sánh theo **`val`** — an toàn cả khi `p`, `q` được truyền vào là **bản sao** (không phải chính đối tượng trong cây), như ví dụ 3 với nút `10` "rời".

In [4]:
p_in_tree = find(root, 5)
p_copy = TreeNode(5)
print("is :", p_in_tree is p_copy, "| val:", p_in_tree.val == p_copy.val)
q_missing = get_node(root, 10)
print("q=10 là nút rời:", q_missing, "| có trong cây?", find(root, 10) is not None)

is : False | val: True
q=10 là nút rời: TreeNode(10) | có trong cây? False


### 3.4 `nonlocal` và giới hạn đệ quy

Hàm lồng nhau muốn **gán lại** biến của hàm ngoài phải khai báo `nonlocal`. Python mặc định giới hạn đệ quy khoảng `1000` tầng; cây lệch `10⁴` nút sẽ vượt giới hạn nếu dùng DFS đệ quy.

In [5]:
def counter_demo():
    total = 0
    def add(x):
        nonlocal total
        total += x
    for x in [1, 2, 3]:
        add(x)
    return total

print("nonlocal:", counter_demo())
print("giới hạn đệ quy:", sys.getrecursionlimit())

nonlocal: 6
giới hạn đệ quy: 1000


### 3.5 BFS với hàng đợi `deque` và bảng cha

BFS duyệt theo tầng bằng `deque`. Ghi `parent[child] = node` khi gặp mỗi nút con ⇒ từ bất kỳ nút nào cũng **đi ngược lên gốc** được mà không cần đệ quy.

In [6]:
parent = {root.val: None}
queue = deque([root])
while queue:
    node = queue.popleft()
    for child in (node.left, node.right):
        if child:
            parent[child.val] = node.val
            queue.append(child)
path, x = [], 4
while x is not None:
    path.append(x); x = parent[x]
print("bảng cha:", parent)
print("đi ngược từ 4 lên gốc:", path)

bảng cha: {3: None, 5: 3, 1: 3, 6: 5, 2: 5, 0: 1, 8: 1, 7: 2, 4: 2}
đi ngược từ 4 lên gốc: [4, 2, 5, 3]


## 4. Ý tưởng nền tảng / trực giác

Làm tay trên cây ví dụ. Viết **đường đi từ gốc** tới từng nút:
- `5`: `3 → 5`
- `1`: `3 → 1`
- `4`: `3 → 5 → 2 → 4`

LCA = nút **cuối cùng** của **tiền tố chung** hai đường đi:
- `(5, 1)`: tiền tố chung `[3]` ⇒ `3`.
- `(5, 4)`: tiền tố chung `[3, 5]` ⇒ `5`.
- `(5, 10)`: không có đường đi tới `10` ⇒ `None`.

Nhìn theo cây con: một nút `x` là LCA ⇔ (cây con của `x` chứa cả `p` lẫn `q`) **và** (không cây con nào của con `x` chứa cả hai). Điểm khác so với bài 236: ta **không được giả định** `p`, `q` có mặt — phải **thấy tận mắt** cả hai.

In [7]:
def path_to(node, val):
    """Đường đi gốc -> val (list giá trị), None nếu không có. Đệ quy — chỉ dùng cho cây nhỏ."""
    if node is None:
        return None
    if node.val == val:
        return [node.val]
    for child in (node.left, node.right):
        sub = path_to(child, val)
        if sub is not None:
            return [node.val] + sub
    return None

for a, b in [(5, 1), (5, 4), (5, 10), (7, 8), (6, 4)]:
    pa, pb = path_to(root, a), path_to(root, b)
    common = []
    if pa and pb:
        for x, y in zip(pa, pb):
            if x != y: break
            common.append(x)
    print(f"({a}, {b}): path {a} = {pa}, path {b} = {pb} -> tiền tố chung {common} -> LCA = {common[-1] if common else None}")

(5, 1): path 5 = [3, 5], path 1 = [3, 1] -> tiền tố chung [3] -> LCA = 3
(5, 4): path 5 = [3, 5], path 4 = [3, 5, 2, 4] -> tiền tố chung [3, 5] -> LCA = 5
(5, 10): path 5 = [3, 5], path 10 = None -> tiền tố chung [] -> LCA = None
(7, 8): path 7 = [3, 5, 2, 7], path 8 = [3, 1, 8] -> tiền tố chung [3] -> LCA = 3
(6, 4): path 6 = [3, 5, 6], path 4 = [3, 5, 2, 4] -> tiền tố chung [3, 5] -> LCA = 5


## 5. Cách 1 — Brute Force / baseline: kiểm tra từng nút

**Ý tưởng:** với mỗi nút `x`, gọi `contains(x, v)` để kiểm tra cây con của `x` có chứa giá trị `v` không. Trong các nút có cây con chứa **cả** `p` và `q`, LCA là nút **sâu nhất**. Nếu không nút nào thỏa (vì thiếu `p` hoặc `q`) ⇒ `None`.

In [8]:
def contains(node, val):
    stack = [node] if node else []
    while stack:
        cur = stack.pop()
        if cur.val == val:
            return True
        if cur.left: stack.append(cur.left)
        if cur.right: stack.append(cur.right)
    return False

def lca_brute(root, p, q):
    best, best_depth = None, -1
    stack = [(root, 0)]
    while stack:
        node, depth = stack.pop()
        if contains(node, p.val) and contains(node, q.val) and depth > best_depth:
            best, best_depth = node, depth
        if node.left: stack.append((node.left, depth + 1))
        if node.right: stack.append((node.right, depth + 1))
    return best

for vals, pv, qv, exp in examples:
    r = build_tree(vals)
    ans = lca_brute(r, get_node(r, pv), get_node(r, qv))
    print(f"p={pv}, q={qv} -> {ans.val if ans else None} (kỳ vọng {exp})")

p=5, q=1 -> 3 (kỳ vọng 3)
p=5, q=4 -> 5 (kỳ vọng 5)
p=5, q=10 -> None (kỳ vọng None)


**Độ phức tạp:** `n` nút, mỗi nút gọi `contains` hai lần tốn `O(kích thước cây con)` ⇒ tổng `O(n · h)` ≤ `O(n²)` (cây lệch). Với `n = 10⁴` cây lệch: `~10⁸` bước — quá chậm. Bộ nhớ `O(n)`.

**Nút thắt:** cùng một cây con bị quét đi quét lại cho từng tổ tiên của nó. Ta muốn mỗi nút **chỉ thăm một lần** và "báo cáo" kết quả lên cha.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** cho mỗi nút `x` trả về `cnt(x)` = số nút mục tiêu (`p` hoặc `q`) nằm trong cây con của `x` — một số trong `{0, 1, 2}`:

`cnt(x) = cnt(x.left) + cnt(x.right) + [x là p hoặc q]`

Duyệt **hậu thứ tự** (con trước, cha sau). Nút **đầu tiên** (sâu nhất) có `cnt(x) == 2` chính là LCA. Vì mọi tổ tiên của nó cũng có `cnt = 2`, ta chỉ ghi nhận **lần đầu**:

- **Nếu** `cnt(x) == 2` **và** chưa có đáp án **thì** `ans = x`.
- **Nếu** duyệt xong mà `ans` vẫn rỗng **thì** thiếu `p` hoặc `q` ⇒ `None`.

Điểm then chốt so với bài 236: **không được dừng sớm** khi gặp `p` (hoặc `q`) — vì nút kia có thể nằm **bên dưới** nó, và ta cần **thấy** nút kia để chắc chắn nó tồn tại. Đây chính là câu trả lời cho follow-up: một lần duyệt, không cần bước kiểm tra tồn tại riêng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — So sánh hai đường đi từ gốc

Tìm đường đi `gốc → p` và `gốc → q` (DFS lặp, lưu cha). Nếu thiếu một trong hai ⇒ `None`. Ngược lại LCA là phần tử cuối của tiền tố chung.

In [9]:
def root_path(root, val):
    """Đường đi (list nút) từ gốc tới nút có giá trị val, lặp bằng stack + bảng cha."""
    parent = {id(root): None}
    stack = [root]
    target = None
    while stack:
        node = stack.pop()
        if node.val == val:
            target = node; break
        for child in (node.left, node.right):
            if child:
                parent[id(child)] = node
                stack.append(child)
    if target is None:
        return None
    path = []
    while target is not None:
        path.append(target); target = parent[id(target)]
    return path[::-1]

def lca_paths(root, p, q):
    pp, pq = root_path(root, p.val), root_path(root, q.val)
    if pp is None or pq is None:
        return None
    ans = None
    for a, b in zip(pp, pq):
        if a is not b:
            break
        ans = a
    return ans

for vals, pv, qv, exp in examples:
    r = build_tree(vals)
    ans = lca_paths(r, get_node(r, pv), get_node(r, qv))
    print(f"p={pv}, q={qv} -> {ans.val if ans else None} (kỳ vọng {exp})")

p=5, q=1 -> 3 (kỳ vọng 3)
p=5, q=4 -> 5 (kỳ vọng 5)
p=5, q=10 -> None (kỳ vọng None)


**Độ phức tạp:** `O(n)` thời gian (hai lần tìm kiếm + so sánh), `O(n)` bộ nhớ.

### 7.2 Cách 3 — LCA của bài 236 + kiểm tra tồn tại

Hàm LCA chuẩn của bài 236: nếu nút là `p` hoặc `q` thì trả về chính nó; nếu hai con đều trả về khác `None` thì nút hiện tại là LCA. Hàm này **giả định** `p`, `q` có mặt. Ta bổ sung: **trước** khi tin kết quả, kiểm tra cả `p` và `q` đều có trong cây.

In [10]:
def lca_236(node, p, q):
    if node is None or node.val == p.val or node.val == q.val:
        return node
    left = lca_236(node.left, p, q)
    right = lca_236(node.right, p, q)
    if left and right:
        return node
    return left or right

def lca_236_checked(root, p, q):
    if not contains(root, p.val) or not contains(root, q.val):
        return None
    return lca_236(root, p, q)

for vals, pv, qv, exp in examples:
    r = build_tree(vals)
    a = lca_236(r, get_node(r, pv), get_node(r, qv))
    b = lca_236_checked(r, get_node(r, pv), get_node(r, qv))
    print(f"p={pv}, q={qv} -> 236 thuần: {a.val if a else None} | có kiểm tra: {b.val if b else None} (kỳ vọng {exp})")

p=5, q=1 -> 236 thuần: 3 | có kiểm tra: 3 (kỳ vọng 3)
p=5, q=4 -> 236 thuần: 5 | có kiểm tra: 5 (kỳ vọng 5)
p=5, q=10 -> 236 thuần: 5 | có kiểm tra: None (kỳ vọng None)


Thấy rõ ở ví dụ 3: LCA 236 thuần trả `5` (vì gặp `5` là dừng, không biết `10` vắng mặt). Có kiểm tra thì đúng `None`.

**Độ phức tạp:** `O(n)` thời gian (ba lượt duyệt), `O(h)` stack đệ quy. Đúng, nhưng **không** trả lời follow-up (vẫn phải kiểm tra tồn tại riêng).

### 7.3 Cách 4 — Một lần DFS hậu thứ tự đếm số nút tìm thấy (follow-up)

Từng đoạn:
1. `dfs(node)` trả về `0/1/2` = số mục tiêu trong cây con.
2. Gọi **cả hai** con trước (không dừng sớm).
3. `cnt = left + right + (node là p hoặc q)`.
4. `cnt == 2` và chưa có đáp án ⇒ ghi `ans = node`.

In [11]:
def lca_count(root, p, q):
    ans = None
    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        cnt = dfs(node.left) + dfs(node.right) + (node.val == p.val or node.val == q.val)
        if cnt == 2 and ans is None:
            ans = node
        return cnt
    dfs(root)
    return ans

class SolutionRecursive:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        ans = None
        def dfs(node):
            nonlocal ans
            if node is None:
                return 0
            cnt = dfs(node.left) + dfs(node.right) + (node.val == p.val or node.val == q.val)
            if cnt == 2 and ans is None:
                ans = node
            return cnt
        dfs(root)
        return ans

for vals, pv, qv, exp in examples:
    r = build_tree(vals)
    a = lca_count(r, get_node(r, pv), get_node(r, qv))
    b = SolutionRecursive().lowestCommonAncestor(r, get_node(r, pv), get_node(r, qv))
    print(f"p={pv}, q={qv} -> {a.val if a else None}, {b.val if b else None} (kỳ vọng {exp})")

p=5, q=1 -> 3, 3 (kỳ vọng 3)
p=5, q=4 -> 5, 5 (kỳ vọng 5)
p=5, q=10 -> None, None (kỳ vọng None)


**Độ phức tạp:** `O(n)` thời gian — mỗi nút thăm đúng một lần; `O(h)` bộ nhớ stack (`h` là chiều cao). **Nhưng** với cây lệch `h = 10⁴` thì Python mặc định sẽ `RecursionError` (xem mục 13). Trên LeetCode, giới hạn đệ quy có thể được nâng, nhưng để chắc chắn ta dùng bản lặp.

### 7.4 Cách 5 — Lặp: BFS dựng bảng cha + tập tổ tiên

1. BFS từ gốc, ghi `parent` cho mọi nút, đồng thời ghi lại nút có giá trị `p.val`, `q.val` nếu gặp.
2. Thiếu một trong hai ⇒ `None`.
3. Đi từ `p` lên gốc, cho mọi tổ tiên (gồm chính `p`) vào tập `anc`.
4. Đi từ `q` lên; nút đầu tiên thuộc `anc` là LCA.

In [12]:
def lca_parent(root, p, q):
    parent = {root: None}
    found = {}
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node.val == p.val or node.val == q.val:
            found[node.val] = node
        for child in (node.left, node.right):
            if child:
                parent[child] = node
                queue.append(child)
    if p.val not in found or q.val not in found:
        return None
    anc = set()
    x = found[p.val]
    while x is not None:
        anc.add(x); x = parent[x]
    y = found[q.val]
    while y not in anc:
        y = parent[y]
    return y

for vals, pv, qv, exp in examples:
    r = build_tree(vals)
    a = lca_parent(r, get_node(r, pv), get_node(r, qv))
    print(f"p={pv}, q={qv} -> {a.val if a else None} (kỳ vọng {exp})")

p=5, q=1 -> 3 (kỳ vọng 3)
p=5, q=4 -> 5 (kỳ vọng 5)
p=5, q=10 -> None (kỳ vọng None)


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (bảng cha). Không đệ quy ⇒ an toàn với mọi hình dạng cây. Đối tượng `TreeNode` mặc định băm theo `id` nên dùng làm khóa `dict`/`set` được.

Ta cũng có thể viết **cách 4 dạng lặp** (hậu thứ tự bằng stack) — đây là lời giải nộp chính ở mục 9: vẫn một lượt duyệt, vẫn trả lời follow-up, lại không đệ quy.

## 8. TRACE từng bước

Trace cách 4 (đếm hậu thứ tự) với `p = 5`, `q = 4`. Thứ tự hậu thứ tự: `6, 7, 4, 2, 5, 0, 8, 1, 3`.

| Nút | cnt trái | cnt phải | là p/q? | cnt | ghi ans? |
|---|---|---|---|---|---|
| 6 | 0 | 0 | không | 0 | |
| 7 | 0 | 0 | không | 0 | |
| 4 | 0 | 0 | **q** | 1 | |
| 2 | 0 | 1 | không | 1 | |
| 5 | 0 | 1 | **p** | 2 | **ans = 5** |
| 0, 8, 1 | … | … | không | 0 | |
| 3 | 2 | 0 | không | 2 | đã có ans, giữ 5 |

In [13]:
def trace_count(values, pv, qv):
    r = build_tree(values)
    p, q = get_node(r, pv), get_node(r, qv)
    ans = None
    def dfs(node, depth):
        nonlocal ans
        if node is None:
            return 0
        l = dfs(node.left, depth + 1)
        rr = dfs(node.right, depth + 1)
        me = int(node.val == p.val or node.val == q.val)
        cnt = l + rr + me
        note = ""
        if cnt == 2 and ans is None:
            ans = node
            note = "  <-- ghi ans"
        print("  " * depth + f"nút {node.val}: trái={l}, phải={rr}, là p/q={me}, cnt={cnt}{note}")
        return cnt
    total = dfs(r, 0)
    print(f"Tổng số mục tiêu thấy được = {total} -> LCA = {ans.val if ans else None}")
    return ans

trace_count(TREE, 5, 4)

    nút 6: trái=0, phải=0, là p/q=0, cnt=0
      nút 7: trái=0, phải=0, là p/q=0, cnt=0
      nút 4: trái=0, phải=0, là p/q=1, cnt=1
    nút 2: trái=0, phải=1, là p/q=0, cnt=1
  nút 5: trái=0, phải=1, là p/q=1, cnt=2  <-- ghi ans
    nút 0: trái=0, phải=0, là p/q=0, cnt=0
    nút 8: trái=0, phải=0, là p/q=0, cnt=0
  nút 1: trái=0, phải=0, là p/q=0, cnt=0
nút 3: trái=2, phải=0, là p/q=0, cnt=2
Tổng số mục tiêu thấy được = 2 -> LCA = 5


TreeNode(5)

In [14]:
trace_count(TREE, 5, 10)

    nút 6: trái=0, phải=0, là p/q=0, cnt=0
      nút 7: trái=0, phải=0, là p/q=0, cnt=0
      nút 4: trái=0, phải=0, là p/q=0, cnt=0
    nút 2: trái=0, phải=0, là p/q=0, cnt=0
  nút 5: trái=0, phải=0, là p/q=1, cnt=1
    nút 0: trái=0, phải=0, là p/q=0, cnt=0
    nút 8: trái=0, phải=0, là p/q=0, cnt=0
  nút 1: trái=0, phải=0, là p/q=0, cnt=0
nút 3: trái=1, phải=0, là p/q=0, cnt=1
Tổng số mục tiêu thấy được = 1 -> LCA = None


Với `q = 10` vắng mặt, gốc chỉ có `cnt = 1` ⇒ không nút nào đạt 2 ⇒ `None`. Lưu ý thuật toán **vẫn duyệt xuống dưới `5`** — nhờ đó nếu `10` nằm dưới `5` ta sẽ thấy.

Trace cách 5 (bảng cha) cho `p = 7`, `q = 8`:

In [15]:
def trace_parent(values, pv, qv):
    r = build_tree(values)
    parent = {r: None}; found = {}
    queue = deque([r])
    while queue:
        node = queue.popleft()
        if node.val in (pv, qv):
            found[node.val] = node
        for child in (node.left, node.right):
            if child:
                parent[child] = node; queue.append(child)
    print("bảng cha:", {k.val: (v.val if v else None) for k, v in parent.items()})
    if pv not in found or qv not in found:
        print("thiếu nút -> None"); return None
    anc, x = [], found[pv]
    while x is not None:
        anc.append(x.val); x = parent[x]
    print(f"tổ tiên của {pv} (gồm chính nó):", anc)
    y = found[qv]
    while y.val not in anc:
        print(f"   {y.val} không thuộc tập tổ tiên -> lên cha {parent[y].val}")
        y = parent[y]
    print(f"   {y.val} thuộc tập tổ tiên -> LCA = {y.val}")
    return y

trace_parent(TREE, 7, 8)

bảng cha: {3: None, 5: 3, 1: 3, 6: 5, 2: 5, 0: 1, 8: 1, 7: 2, 4: 2}
tổ tiên của 7 (gồm chính nó): [7, 2, 5, 3]
   8 không thuộc tập tổ tiên -> lên cha 1
   1 không thuộc tập tổ tiên -> lên cha 3
   3 thuộc tập tổ tiên -> LCA = 3


TreeNode(3)

## 9. Code nộp LeetCode

Lời giải chính: **hậu thứ tự lặp** (một lượt duyệt, đếm số mục tiêu, không đệ quy — an toàn với cây lệch `10⁴` nút). Mỗi phần tử stack là `(node, đã_xử_lý_con)`; khi lấy ra lần thứ hai thì tính `cnt` từ kết quả của hai con lưu trong `dict`.

In [16]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, x):
#         self.val = x
#         self.left = None
#         self.right = None

class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        targets = (p.val, q.val)
        cnt = {None: 0}                 # cnt[node] = số mục tiêu trong cây con
        stack = [(root, False)]
        while stack:
            node, done = stack.pop()
            if node is None:
                continue
            if not done:
                stack.append((node, True))      # xử lý nút sau hai con
                stack.append((node.right, False))
                stack.append((node.left, False))
            else:
                c = cnt[node.left] + cnt[node.right] + (node.val in targets)
                if c == 2:
                    return node                 # nút đầu tiên đạt 2 theo hậu thứ tự là sâu nhất
                cnt[node] = c
        return None

for vals, pv, qv, exp in examples:
    r = build_tree(vals)
    ans = Solution().lowestCommonAncestor(r, get_node(r, pv), get_node(r, qv))
    assert (ans.val if ans else None) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


Ở bản lặp ta **trả về ngay** khi `c == 2`: trong hậu thứ tự, mọi nút trong cây con của `x` được xử lý **trước** `x`, nên nút đầu tiên đạt `2` không có hậu duệ nào đạt `2` ⇒ nó là nút sâu nhất. Và khi `c == 2` thì cả hai mục tiêu **đã được nhìn thấy** — tồn tại chắc chắn.

## 10. Vì sao thuật toán đúng

Gọi `S(x)` là tập nút trong cây con gốc `x`, `cnt(x) = |S(x) ∩ {p, q}|` (đếm những mục tiêu **thực sự có** trong cây).

1. **Tính đúng của `cnt`:** `S(x) = {x} ∪ S(x.left) ∪ S(x.right)` (rời nhau) ⇒ `cnt(x) = [x ∈ {p,q}] + cnt(left) + cnt(right)`. Hậu thứ tự bảo đảm hai con được tính trước.
2. **Nếu `p` hoặc `q` vắng mặt:** `cnt(x) ≤ 1` với mọi `x` ⇒ không nút nào đạt 2 ⇒ trả `None`. Đúng yêu cầu.
3. **Nếu cả hai có mặt:** tập `A = {x : cnt(x) = 2}` là tập **tổ tiên chung** của `p` và `q`. Tập này là một **đường** từ gốc xuống (nếu `x ∈ A` thì cha của `x` cũng thuộc `A`), nút sâu nhất của nó là LCA theo định nghĩa. Trong hậu thứ tự, nút sâu nhất của một đường được thăm **đầu tiên** (con trước cha) ⇒ lần đầu gặp `cnt = 2` chính là LCA.
4. **Không dừng sớm:** ta không bao giờ bỏ qua cây con của `p` hoặc `q`, nên trường hợp "`q` nằm dưới `p`" vẫn được đếm đủ (ví dụ 2: `cnt(5) = 1 + 1 = 2`).

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là chiều cao cây (`log n ≤ h ≤ n`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn `contains` | `O(n·h)` (xấu nhất `O(n²)`) | `O(n)` | quét lại cây con nhiều lần |
| 2. Hai đường đi | `O(n)` | `O(n)` | 2 lượt tìm kiếm |
| 3. LCA 236 + kiểm tra | `O(n)` | `O(h)` stack | 3 lượt duyệt, không trả lời follow-up |
| 4. Đếm hậu thứ tự (đệ quy) | `O(n)` | `O(h)` stack | 1 lượt; tràn đệ quy khi `h` lớn |
| 5. BFS + bảng cha | `O(n)` | `O(n)` | lặp, an toàn |
| Nộp: hậu thứ tự lặp | `O(n)` | `O(n)` | 1 lượt, lặp, trả lời follow-up |

Đo thời gian trên cây ngẫu nhiên cân đối và cây lệch `10⁴` nút (chỉ chạy các bản lặp trên cây lệch):

In [17]:
def random_tree(n, rnd, vals=None):
    """Cây nhị phân ngẫu nhiên n nút, giá trị khác nhau; gắn mỗi nút mới vào một chỗ trống ngẫu nhiên."""
    vals = vals or rnd.sample(range(-10**9, 10**9), n)
    nodes = [TreeNode(vals[0])]
    slots = [(nodes[0], 'L'), (nodes[0], 'R')]
    for v in vals[1:]:
        k = rnd.randrange(len(slots))
        parent_node, side = slots[k]
        slots[k] = slots[-1]; slots.pop()
        child = TreeNode(v)
        setattr(parent_node, 'left' if side == 'L' else 'right', child)
        nodes.append(child)
        slots.append((child, 'L')); slots.append((child, 'R'))
    return nodes[0], nodes

def skewed_tree(n):
    root = TreeNode(0); cur = root
    for v in range(1, n):
        cur.right = TreeNode(v); cur = cur.right
    return root

rnd = random.Random(1644)
R, nodes = random_tree(10**4, rnd)
p, q = rnd.sample(nodes, 2)
for name, fn in [("paths", lca_paths), ("count(rec)", lca_count), ("parent", lca_parent), ("Solution", Solution().lowestCommonAncestor)]:
    t0 = time.perf_counter(); a = fn(R, p, q); dt = time.perf_counter() - t0
    print(f"cây ngẫu nhiên  {name:<11}: LCA={a.val}  {dt * 1000:6.1f} ms")
S = skewed_tree(10**4)
p, q = find(S, 9000), find(S, 9999)
for name, fn in [("parent", lca_parent), ("Solution", Solution().lowestCommonAncestor)]:
    t0 = time.perf_counter(); a = fn(S, p, q); dt = time.perf_counter() - t0
    print(f"cây lệch 10^4   {name:<11}: LCA={a.val}  {dt * 1000:6.1f} ms")

cây ngẫu nhiên  paths      : LCA=-978431990     3.8 ms
cây ngẫu nhiên  count(rec) : LCA=-978431990     1.4 ms
cây ngẫu nhiên  parent     : LCA=-978431990     2.0 ms
cây ngẫu nhiên  Solution   : LCA=-978431990     0.9 ms
cây lệch 10^4   parent     : LCA=9000     2.1 ms
cây lệch 10^4   Solution   : LCA=9000     1.7 ms


## 12. Bộ test

Gồm: ví dụ đề; edge case (cây 1 nút, `p` là gốc, `p` là tổ tiên của `q`, cả hai vắng mặt, chỉ `p` vắng mặt); cây ngẫu nhiên nhỏ có seed cố định so với brute force (cả trường hợp nút vắng mặt); và cây lệch lớn cho các bản lặp.

In [18]:
ALL = {
    "brute": lca_brute,
    "paths": lca_paths,
    "236+check": lca_236_checked,
    "count": lca_count,
    "parent": lca_parent,
    "SolRec": lambda r, p, q: SolutionRecursive().lowestCommonAncestor(r, p, q),
    "Solution": lambda r, p, q: Solution().lowestCommonAncestor(r, p, q),
}

def check(r, p, q, exp="skip"):
    res = {name: fn(r, p, q) for name, fn in ALL.items()}
    vals = {name: (a.val if a else None) for name, a in res.items()}
    assert len(set(vals.values())) == 1, vals
    a = res["Solution"]
    if a is not None:
        assert find(r, a.val) is a, "đáp án phải là nút thật trong cây"
    if exp != "skip":
        assert vals["Solution"] == exp, (vals, exp)
    return vals["Solution"]

for vals_, pv, qv, exp in examples:
    r = build_tree(vals_)
    check(r, get_node(r, pv), get_node(r, qv), exp)
print("Ví dụ đề: OK")

r = build_tree([1])
check(r, get_node(r, 1), get_node(r, 2), None)          # 1 nút, q vắng
check(r, get_node(r, 7), get_node(r, 2), None)          # cả hai vắng
r = build_tree(TREE)
check(r, get_node(r, 3), get_node(r, 4), 3)             # p là gốc
check(r, get_node(r, 2), get_node(r, 7), 2)             # p là cha của q
check(r, get_node(r, 7), get_node(r, 4), 2)
check(r, get_node(r, 6), get_node(r, 8), 3)
check(r, get_node(r, 0), get_node(r, 8), 1)
check(r, get_node(r, 99), get_node(r, 4), None)         # chỉ p vắng
r = build_tree([1, 2, None, 3, None, 4])                 # cây lệch trái
check(r, get_node(r, 4), get_node(r, 2), 2)
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [19]:
rnd = random.Random(2025)
for _ in range(2500):
    n = rnd.randint(1, 15)
    vals_ = rnd.sample(range(-20, 21), n)
    r, nodes = random_tree(n, rnd, vals_)
    pv, qv = rnd.sample(range(-25, 26), 2)              # có thể vắng mặt
    if rnd.random() < 0.6 and n >= 2:
        pv, qv = (x.val for x in rnd.sample(nodes, 2))  # thường chọn nút có mặt
    check(r, get_node(r, pv), get_node(r, qv))
print("2500 cây ngẫu nhiên nhỏ: OK")

S = skewed_tree(10**4)
assert Solution().lowestCommonAncestor(S, find(S, 5), find(S, 9999)).val == 5
assert lca_parent(S, find(S, 5), find(S, 9999)).val == 5
assert Solution().lowestCommonAncestor(S, find(S, 5), TreeNode(10**4)) is None
print("Cây lệch 10^4 nút (bản lặp): OK")

2500 cây ngẫu nhiên nhỏ: OK


Cây lệch 10^4 nút (bản lặp): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng nguyên lời giải bài 236

In [20]:
r = build_tree(TREE)
a = lca_236(r, get_node(r, 5), get_node(r, 10))
print("236 thuần với q=10 vắng mặt:", a, "| đúng:", Solution().lowestCommonAncestor(r, get_node(r, 5), get_node(r, 10)))

236 thuần với q=10 vắng mặt: TreeNode(5) | đúng: None


**Vì sao sai:** lời giải 236 dừng ngay khi gặp `p` và **giả định** `q` ở đâu đó trong cây. Khi `q` không tồn tại, nó vẫn trả `p`. **Sửa:** kiểm tra tồn tại (cách 3) hoặc đếm số mục tiêu thấy được (cách 4).

### Lỗi 2 — Đếm nhưng vẫn "dừng sớm" khi gặp `p`/`q`

In [21]:
def lca_bug_early_stop(root, p, q):
    ans = None
    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        if node.val == p.val or node.val == q.val:
            return 1                         # SAI: không xuống tiếp hai con
        cnt = dfs(node.left) + dfs(node.right)
        if cnt == 2 and ans is None:
            ans = node
        return cnt
    dfs(root)
    return ans

r = build_tree(TREE)
a = lca_bug_early_stop(r, get_node(r, 5), get_node(r, 4))
b = Solution().lowestCommonAncestor(r, get_node(r, 5), get_node(r, 4))
print("dừng sớm (p=5, q=4):", a, "| đúng:", b)

dừng sớm (p=5, q=4): None | đúng: TreeNode(5)


**Vì sao sai:** `4` nằm **dưới** `5`; dừng ở `5` thì không bao giờ đếm được `4` ⇒ không nút nào đạt 2 ⇒ `None` (sai, đáp án là `5`). **Sửa:** luôn duyệt hai con rồi mới cộng `[node là mục tiêu]`.

### Lỗi 3 — Ghi đè đáp án ở mọi nút có `cnt == 2`

In [22]:
def lca_bug_overwrite(root, p, q):
    ans = None
    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        cnt = dfs(node.left) + dfs(node.right) + (node.val == p.val or node.val == q.val)
        if cnt == 2:
            ans = node                       # SAI: tổ tiên phía trên ghi đè LCA thật
        return cnt
    dfs(root)
    return ans

r = build_tree(TREE)
print("ghi đè (p=7, q=4):", lca_bug_overwrite(r, get_node(r, 7), get_node(r, 4)),
      "| đúng:", Solution().lowestCommonAncestor(r, get_node(r, 7), get_node(r, 4)))

ghi đè (p=7, q=4): TreeNode(3) | đúng: TreeNode(2)


**Vì sao sai:** mọi tổ tiên của LCA cũng có `cnt = 2`; hậu thứ tự xử lý chúng **sau** nên ghi đè lên tới gốc. **Sửa:** chỉ ghi khi `ans is None` (hoặc trả về ngay như bản lặp).

### Lỗi 4 — DFS đệ quy trên cây lệch sâu

In [23]:
S = skewed_tree(10**4)
try:
    lca_count(S, find(S, 5), find(S, 9999))
    print("Đệ quy chạy được (giới hạn đệ quy đã được nâng)")
except RecursionError:
    print("RecursionError với cây lệch 10^4 nút (giới hạn =", sys.getrecursionlimit(), ")")
print("Bản lặp:", Solution().lowestCommonAncestor(S, find(S, 5), find(S, 9999)))

RecursionError với cây lệch 10^4 nút (giới hạn = 1000 )
Bản lặp: TreeNode(5)


**Vì sao sai:** chiều cao `10⁴` > giới hạn đệ quy mặc định `1000`. **Sửa:** dùng bản lặp (stack/BFS), hoặc `sys.setrecursionlimit` (có rủi ro tràn stack C, không khuyến khích).

### Lỗi 5 — So sánh `node is p` khi `p` có thể là bản sao

Nếu đề/driver truyền `p` là một **đối tượng khác** có cùng giá trị (như nút `10` "rời" ở ví dụ 3 — hoặc bất kỳ bản sao nào), so sánh `is` sẽ không bao giờ khớp:

In [24]:
def lca_bug_identity(root, p, q):
    ans = None
    def dfs(node):
        nonlocal ans
        if node is None:
            return 0
        cnt = dfs(node.left) + dfs(node.right) + (node is p or node is q)
        if cnt == 2 and ans is None:
            ans = node
        return cnt
    dfs(root)
    return ans

r = build_tree(TREE)
print("so sánh is với bản sao TreeNode(5), TreeNode(1):", lca_bug_identity(r, TreeNode(5), TreeNode(1)))
print("so sánh val:", Solution().lowestCommonAncestor(r, TreeNode(5), TreeNode(1)))

so sánh is với bản sao TreeNode(5), TreeNode(1): None
so sánh val: TreeNode(3)


**Sửa:** vì giá trị là duy nhất, so sánh theo `val` an toàn hơn. (Nếu chắc chắn driver truyền chính nút trong cây thì `is` cũng đúng.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "LCA trong cây nhị phân" ⇒ DFS **hậu thứ tự**, mỗi nút tổng hợp kết quả hai con.
- "Có thể **không tồn tại**" ⇒ không được dừng sớm; trả về **số lượng tìm thấy** thay vì "nút tìm thấy".
- Biến "tìm nút đầu tiên thỏa điều kiện theo hậu thứ tự" thành `if cond and ans is None: ans = node` — hoặc `return` ngay trong bản lặp.
- Cây tới `10⁴` nút trong Python ⇒ nghĩ đến cây **lệch** và **giới hạn đệ quy**; có sẵn mẫu hậu thứ tự lặp `(node, done)`.
- Bảng cha + tập tổ tiên là "dao đa năng" cho mọi biến thể LCA (cả cây không nhị phân).

## 15. Biến thể của bài

1. **LeetCode 236 — LCA of a Binary Tree:** `p`, `q` chắc chắn tồn tại ⇒ được dừng sớm.
2. **LeetCode 235 — LCA of a BST:** dùng tính chất BST: đi xuống trái/phải theo giá trị, `O(h)`.
3. **LeetCode 1650 — LCA III:** nút có con trỏ `parent`, không có gốc ⇒ hai con trỏ như tìm giao điểm hai danh sách liên kết.
4. **LeetCode 1676 — LCA IV:** LCA của **nhiều** nút ⇒ đếm/kiểm tra tập mục tiêu.
5. **LeetCode 1123 / 865 — LCA of Deepest Leaves:** hậu thứ tự trả về (độ sâu, nút).
6. **Nhiều truy vấn LCA trên cùng cây:** nhảy nhị phân (binary lifting) `O(log n)` mỗi truy vấn hoặc Euler tour + RMQ.
7. **Khoảng cách giữa hai nút:** `depth(p) + depth(q) − 2·depth(LCA)`.

In [25]:
def lca_bst(root, p, q):
    """LCA trên cây BST (giả sử cả hai tồn tại)."""
    node = root
    while node:
        if p.val < node.val and q.val < node.val:
            node = node.left
        elif p.val > node.val and q.val > node.val:
            node = node.right
        else:
            return node

bst = build_tree([6, 2, 8, 0, 4, 7, 9, None, None, 3, 5])
print("LCA BST(2, 8) =", lca_bst(bst, TreeNode(2), TreeNode(8)).val, "| LCA BST(3, 5) =", lca_bst(bst, TreeNode(3), TreeNode(5)).val)
print("đối chiếu Solution:", Solution().lowestCommonAncestor(bst, TreeNode(2), TreeNode(8)).val,
      Solution().lowestCommonAncestor(bst, TreeNode(3), TreeNode(5)).val)

LCA BST(2, 8) = 6 | LCA BST(3, 5) = 4
đối chiếu Solution: 6 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | nút sâu nhất có cây con chứa cả hai | `O(n²)` | `O(n)` | đối chiếu |
| 2. Hai đường đi | tiền tố chung của 2 đường từ gốc | `O(n)` | `O(n)` | dễ nghĩ, dễ giải thích |
| 3. 236 + kiểm tra | kiểm tra tồn tại rồi LCA chuẩn | `O(n)` | `O(h)` | khi đã thuộc 236 |
| 4. Đếm hậu thứ tự | `cnt ∈ {0,1,2}`, ghi nút đầu tiên đạt 2 | `O(n)` | `O(h)` | **follow-up**, một lượt |
| 5. Bảng cha | BFS + tập tổ tiên | `O(n)` | `O(n)` | cần lặp; nhiều biến thể |
| Nộp | hậu thứ tự lặp đếm | `O(n)` | `O(n)` | an toàn mọi hình dạng cây |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Trên cây ví dụ, LCA của `(6, 4)`, `(0, 1)` và `(8, 11)` là gì?

**Bài 2 (tính tay):** Với `p = 7`, `q = 0`, ghi giá trị `cnt` của mọi nút theo hậu thứ tự và cho biết nút nào được ghi làm đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `distance(root, a, b)`: khoảng cách (số cạnh) giữa hai nút giá trị `a`, `b`; trả `-1` nếu một trong hai vắng mặt.

**Bài 4 (tự viết lại):** Viết lại cách 4 bằng một hàm đệ quy **trả về tuple** `(cnt, lca)` — không dùng `nonlocal`.

**Bài 5 (mini challenge):** Viết `lca_many(root, values)` trả về LCA của **tập** giá trị `values` (có thể có giá trị vắng mặt ⇒ trả `None`), bằng hậu thứ tự lặp đếm số mục tiêu.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def distance(root, a, b):
    # TODO: tìm LCA (dùng Solution), tính độ sâu bằng BFS; khoảng cách = da + db - 2*dl
    pass

# Bài 4 — TODO
def lca_tuple(root, pv, qv):
    # TODO: def go(node) -> (cnt, lca_or_None); lca của con nào khác None thì truyền lên
    pass

# Bài 5 — TODO
def lca_many(root, values):
    # TODO: như Solution nhưng targets = set(values), so sánh c == len(targets)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `6` và `4` đều dưới `5`.
- Mức 2: `0` là con của `1`.
- Mức 3: `11` không có trong cây.

**Bài 2**
- Mức 1: `7` ở nhánh trái của gốc, `0` ở nhánh phải.
- Mức 2: `cnt(5) = 1`, `cnt(1) = 1`.
- Mức 3: Chỉ gốc `3` đạt `cnt = 2`.

**Bài 3**
- Mức 1: Khoảng cách đi qua LCA.
- Mức 2: BFS từ gốc lưu `depth[val]`.
- Mức 3: Nếu `Solution` trả `None` ⇒ `-1`; trường hợp `a == b` ⇒ `0`.

**Bài 4**
- Mức 1: `go(None) = (0, None)`.
- Mức 2: Nếu con nào đã có `lca` thì trả luôn `(2, lca đó)`.
- Mức 3: Nếu `cnt == 2` lần đầu ở nút hiện tại thì `lca = node`.

**Bài 5**
- Mức 1: Thay "2" bằng `k = len(set(values))`.
- Mức 2: Giá trị vắng mặt ⇒ không nút nào đạt `k`.
- Mức 3: Trường hợp `k = 1` ⇒ LCA là chính nút đó (nếu có).

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
r = build_tree(TREE)
res1 = []
for a, b in [(6, 4), (0, 1), (8, 11)]:
    x = Solution().lowestCommonAncestor(r, get_node(r, a), get_node(r, b))
    res1.append(x.val if x else None)
print("Bài 1:", res1)
assert res1 == [5, 1, None]

# Bài 2
ans2 = trace_count(TREE, 7, 0)
assert ans2.val == 3

Bài 1: [5, 1, None]
    nút 6: trái=0, phải=0, là p/q=0, cnt=0
      nút 7: trái=0, phải=0, là p/q=1, cnt=1
      nút 4: trái=0, phải=0, là p/q=0, cnt=0
    nút 2: trái=1, phải=0, là p/q=0, cnt=1
  nút 5: trái=0, phải=1, là p/q=0, cnt=1
    nút 0: trái=0, phải=0, là p/q=1, cnt=1
    nút 8: trái=0, phải=0, là p/q=0, cnt=0
  nút 1: trái=1, phải=0, là p/q=0, cnt=1
nút 3: trái=1, phải=1, là p/q=0, cnt=2  <-- ghi ans
Tổng số mục tiêu thấy được = 2 -> LCA = 3


In [28]:
# Bài 3
def distance_sol(root, a, b):
    l = Solution().lowestCommonAncestor(root, TreeNode(a), TreeNode(b)) if a != b else find(root, a)
    if l is None:
        return -1
    depth = {root.val: 0}
    queue = deque([root])
    while queue:
        node = queue.popleft()
        for child in (node.left, node.right):
            if child:
                depth[child.val] = depth[node.val] + 1
                queue.append(child)
    return depth[a] + depth[b] - 2 * depth[l.val]

r = build_tree(TREE)
print("Bài 3: d(7,8) =", distance_sol(r, 7, 8), "| d(5,4) =", distance_sol(r, 5, 4), "| d(5,10) =", distance_sol(r, 5, 10))
assert distance_sol(r, 7, 8) == 5 and distance_sol(r, 5, 4) == 2 and distance_sol(r, 5, 10) == -1 and distance_sol(r, 4, 4) == 0

# Bài 4
def lca_tuple_sol(root, pv, qv):
    def go(node):
        if node is None:
            return 0, None
        cl, al = go(node.left)
        if al is not None:
            return 2, al
        cr, ar = go(node.right)
        if ar is not None:
            return 2, ar
        c = cl + cr + (node.val in (pv, qv))
        return c, (node if c == 2 else None)
    return go(root)[1]

rnd = random.Random(44)
for _ in range(1500):
    n = rnd.randint(1, 14)
    vals_ = rnd.sample(range(30), n)
    r, nodes = random_tree(n, rnd, vals_)
    pv, qv = rnd.sample(range(32), 2)
    exp = lca_brute(r, TreeNode(pv), TreeNode(qv))
    got = lca_tuple_sol(r, pv, qv)
    assert (got.val if got else None) == (exp.val if exp else None)
print("Bài 4 OK")

Bài 3: d(7,8) = 5 | d(5,4) = 2 | d(5,10) = -1
Bài 4 OK


In [29]:
# Bài 5
def lca_many_sol(root, values):
    targets = set(values)
    k = len(targets)
    cnt = {None: 0}
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if node is None:
            continue
        if not done:
            stack.append((node, True)); stack.append((node.right, False)); stack.append((node.left, False))
        else:
            c = cnt[node.left] + cnt[node.right] + (node.val in targets)
            if c == k:
                return node
            cnt[node] = c
    return None

def lca_many_brute(root, values):
    best, best_d = None, -1
    stack = [(root, 0)]
    while stack:
        node, d = stack.pop()
        if all(contains(node, v) for v in values) and d > best_d:
            best, best_d = node, d
        for ch in (node.left, node.right):
            if ch: stack.append((ch, d + 1))
    return best

rnd = random.Random(66)
for _ in range(1500):
    n = rnd.randint(1, 14)
    vals_ = rnd.sample(range(30), n)
    r, nodes = random_tree(n, rnd, vals_)
    values = rnd.sample(range(32), rnd.randint(1, 4))
    a, b = lca_many_sol(r, values), lca_many_brute(r, values)
    assert (a.val if a else None) == (b.val if b else None), (values,)
r = build_tree(TREE)
print("Bài 5 OK | lca_many([7, 6, 4]) =", lca_many_sol(r, [7, 6, 4]).val, "| lca_many([7, 8]) =", lca_many_sol(r, [7, 8]).val)

Bài 5 OK | lca_many([7, 6, 4]) = 5 | lca_many([7, 8]) = 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template hậu thứ tự lặp "đếm mục tiêu trong cây con":**

```python
cnt = {None: 0}
stack = [(root, False)]
while stack:
    node, done = stack.pop()
    if node is None:
        continue
    if not done:
        stack.append((node, True))
        stack.append((node.right, False))
        stack.append((node.left, False))
    else:
        c = cnt[node.left] + cnt[node.right] + (node.val in targets)
        if c == len(targets):
            return node          # nút sâu nhất chứa đủ mục tiêu
        cnt[node] = c
return None                      # thiếu mục tiêu
```

**Template đệ quy (cây nông):** `cnt = dfs(l) + dfs(r) + is_target(node); if cnt == 2 and ans is None: ans = node`.

**Checklist:**
- [ ] Không dừng sớm khi gặp `p`/`q` (nút kia có thể ở dưới)?
- [ ] Trả `None` khi thiếu `p` hoặc `q`?
- [ ] Chỉ ghi đáp án **lần đầu** đạt 2 (không để tổ tiên ghi đè)?
- [ ] So sánh theo `val` (duy nhất) — an toàn khi `p`, `q` là bản sao?
- [ ] Cây lệch `10⁴` nút: dùng bản lặp hoặc chắc chắn giới hạn đệ quy đủ lớn?
- [ ] Đã đối chiếu với brute force trên cây ngẫu nhiên có nút vắng mặt?
- [ ] Chữ ký đúng: `def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode'`?